**Задание 1.**
Реализуйте базовый класс Account, который моделирует поведение
банковского счёта.

Этот класс должен не только выполнять базовые
операции, но и вести детальный учёт всех действий, а также предоставлять
аналитику по истории операций.

**Этап 1.** Реализация базового класса Account

Класс должен быть инициализирован с параметрами:
*   account_holder (str) — имя владельца счёта;
*   balance (float, по умолчанию 0) — начальный баланс счёта, не может быть отрицательным.

Атрибуты:
*   _account_counter — приватный атрибут для хранения количества созданных счетов. Отсчет начинается с 1000;
*   holder — хранит имя владельца;
*   account_number — хранит номер счёта;
*   _balance — приватный атрибут для хранения текущего баланса;
*   operations_history — список или другая структура для хранения
истории операций.

**Важно**: каждая операция должна храниться не просто как число, а как
структурированная информация, например, словарь, кортеж или класс.
Минимальный набор данных для операции: тип операции ('deposit' или
'withdraw'), сумма, дата и время операции, текущий баланс после операции,
статус ('success' или 'fail').

In [ ]:
import datetime
import re

class Account:
    _account_counter = 1000

    def __init__(self, account_holder: str, balance: float = 0):
        if not self._is_valid_name(account_holder):
            raise ValueError("Имя владельца должно быть в формате 'Имя Фамилия' с заглавных букв, кириллицей или латиницей.")
        if balance < 0:
            raise ValueError("Начальный баланс не может быть отрицательным.")

        self.holder = account_holder
        self.account_number = f"ACC-{Account._account_counter:04d}"
        Account._account_counter += 1
        self._balance = balance
        self.operations_history = []

    @staticmethod
    def _is_valid_name(name: str) -> bool:
        # Проверка формата "Имя Фамилия" с заглавных букв, кириллица или латиница
        pattern = r"^[А-ЯЁA-Z][а-яёa-z]+\s[А-ЯЁA-Z][а-яёa-z]+$"
        return bool(re.match(pattern, name.strip()))

    def deposit(self, amount: float):
        if amount <= 0:
            raise ValueError("Сумма пополнения должна быть положительной.")

        self._balance += amount
        operation_info = {
            'type': 'deposit',
            'amount': amount,
            'timestamp': datetime.datetime.now(),
            'balance_after': self._balance,
            'status': 'success'
        }
        self.operations_history.append(operation_info)

    def withdraw(self, amount: float):
        if amount <= 0:
            raise ValueError("Сумма снятия должна быть положительной.")

        status = 'success' if amount <= self._balance else 'fail'
        operation_info = {
            'type': 'withdraw',
            'amount': amount,
            'timestamp': datetime.datetime.now(),
            'balance_after': self._balance,
            'status': status
        }
        self.operations_history.append(operation_info)

        if status == 'success':
            self._balance -= amount

    def get_balance(self):
        return self._balance

    def get_history(self):
        return self.operations_history

    def get_large_transactions(self, n: int = 5):
        sorted_ops = sorted(
            [op for op in self.operations_history if op['status'] == 'success'],
            key=lambda x: x['amount'],
            reverse=True
        )
        return sorted_ops[:n]

    def __repr__(self):
        return f"Account(account_number='{self.account_number}', holder='{self.holder}', balance={self._balance})"

    def __str__(self):
        return f"Счёт {self.account_number} на имя {self.holder}, баланс: {self._balance:.2f} руб."

In [ ]:
class CheckingAccount(Account):
    account_type = "Checking"

    def __init__(self, account_holder: str, balance: float = 0):
        super().__init__(account_holder, balance)


class SavingsAccount(Account):
    account_type = "Savings"

    def __init__(self, account_holder: str, balance: float = 0):
        super().__init__(account_holder, balance)

    def apply_interest(self, rate: float):
        if rate < 0:
            raise ValueError("Процентная ставка не может быть отрицательной.")
        interest = self._balance * (rate / 100)
        self._balance += interest
        operation_info = {
            'type': 'interest',
            'amount': interest,
            'timestamp': datetime.datetime.now(),
            'balance_after': self._balance,
            'status': 'success'
        }
        self.operations_history.append(operation_info)

    def withdraw(self, amount: float):
        if amount <= 0:
            raise ValueError("Сумма снятия должна быть положительной.")

        if amount > self._balance * 0.5:
            status = 'fail'
            operation_info = {
                'type': 'withdraw',
                'amount': amount,
                'timestamp': datetime.datetime.now(),
                'balance_after': self._balance,
                'status': status
            }
            self.operations_history.append(operation_info)
            return

        self._balance -= amount
        operation_info = {
            'type': 'withdraw',
            'amount': amount,
            'timestamp': datetime.datetime.now(),
            'balance_after': self._balance,
            'status': 'success'
        }
        self.operations_history.append(operation_info)